In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from vascx.fundus.loader import RetinaLoader
from vascx.fundus.features.base import LayerFeature
from vascx.fundus.features.base import VesselsLayerFeature
from vascx.fundus.features.base import RetinaFeature
from vascx.fundus.features.bifurcation_angles import BifurcationAngles
from vascx.fundus.features.caliber import Caliber
from vascx.fundus.features.temporal_angles import TemporalAngle
from vascx.fundus.features.cre import CRE, CREMode
from vascx.fundus.features.tortuosity import Tortuosity, TortuosityMode
from vascx.fundus.features.sparsity import Sparsity, SparsityMode
from vascx.fundus.features.vascular_densities import VascularDensity
from rtnls_enface.grids.hemifields import HemifieldField
from vascx.shared.aggregators import LengthWeightedAggregator, median, mean
from vascx.fundus.feature_sets.full_v3 import (
    ELLIPSE_FULL,
    DISC_FULL,
    ETDRS_FULL,
    HMF_SUP,
    HMF_INF,
)

plt.rcParams.update(
    {
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.dpi": 300,
        "font.size": 10,
    }
)

In [ ]:
ds_path = Path("../../samples/fundus")
figs_path = Path("/home/jose/vessels/paper_vascx/figures_hq")
fig_dpi = 300
ncols = 3
figs_path.mkdir(parents=True, exist_ok=True)

loader = RetinaLoader.from_folder(ds_path)

# Pick the three images used in every column (one image per column, all rows)
sample_ids = [
    "CHASEDB1_08L",
    "CHASEDB1_12R",
    "CHASEDB1_01L",
]
assert len(sample_ids) == ncols
for sample_id in sample_ids:
    assert sample_id in loader.indices, f"Unknown sample ID: {sample_id}"

print("Available samples:", loader.indices)
print("Using samples:", sample_ids)

In [ ]:
def plot_feature(ax, feature, sample_id: str) -> None:
    """Plot a single feature on the given axis for one sample."""
    retina = loader.by_id(sample_id)
    if isinstance(feature, LayerFeature):
        feature.plot(ax=ax, layer=retina.veins)
    elif isinstance(feature, VesselsLayerFeature):
        feature.plot(ax=ax, layer=retina.vessels)
    elif isinstance(feature, RetinaFeature):
        feature.plot(ax=ax, retina=retina)
    else:
        raise ValueError(f"Unknown feature type: {type(feature)}")


def plot_biomarker_grid(
    biomarker_rows: list[tuple[str, list]],
    ncols: int = 3,
    dpi: int = 100,
    label_width: float = 0.12,
):
    """Plot biomarkers in a grid with one row per biomarker and row labels."""
    nrows = len(biomarker_rows)
    fig = plt.figure(
        figsize=(ncols * 4 + 0.5, nrows * 4),
        dpi=dpi,
    )
    width_ratios = [label_width] + [1.0] * ncols
    gs = fig.add_gridspec(
        nrows,
        ncols + 1,
        width_ratios=width_ratios,
        wspace=0.05,
        hspace=0.08,
    )

    for row_idx, (label, features) in enumerate(biomarker_rows):
        label_ax = fig.add_subplot(gs[row_idx, 0])
        label_ax.axis("off")
        label_ax.text(
            0.5,
            0.5,
            label,
            ha="center",
            va="center",
            rotation=90,
            fontsize=9,
            transform=label_ax.transAxes,
        )

        for col_idx, feature in enumerate(features):
            ax = fig.add_subplot(gs[row_idx, col_idx + 1])
            ax.set_xticks([])
            ax.set_yticks([])
            sample_id = sample_ids[col_idx]
            plot_feature(ax, feature, sample_id)

    return fig

In [ ]:
# Three varied configurations per biomarker, copied from feature_plots.ipynb
biomarker_rows_part1 = [
    (
        "Vascular\nDensity",
        [
            VascularDensity(grid_field=HMF_INF),
            VascularDensity(grid_field=ELLIPSE_FULL),
            VascularDensity(grid_field=ETDRS_FULL),
        ],
    ),
    (
        "Bifurcation\nAngles",
        [
            BifurcationAngles(aggregator=mean),
            BifurcationAngles(grid_field=ELLIPSE_FULL, aggregator=median),
            BifurcationAngles(grid_field=HMF_SUP, aggregator=mean),
        ],
    ),
    (
        "Calibers",
        [
            Caliber(grid_field=DISC_FULL, aggregator=LengthWeightedAggregator()),
            Caliber(grid_field=DISC_FULL, aggregator=median),
            Caliber(aggregator=median),
        ],
    ),
    (
        "Tortuosity",
        [
            Tortuosity(grid_field=DISC_FULL),
            Tortuosity(
                mode=TortuosityMode.Vessels,
                aggregator=LengthWeightedAggregator(),
            ),
            Tortuosity(aggregator=median),
        ],
    ),
]

biomarker_rows_part2 = [
    (
        "CRE",
        [
            CRE(CREMode.Temporal),
            CRE(CREMode.Temporal, hemifield=HemifieldField.Superior),
            CRE(CREMode.Nasal),
        ],
    ),
    (
        "Temporal\nAngle",
        [
            TemporalAngle(),
            TemporalAngle(),
            TemporalAngle(),
        ],
    ),
    (
        "Sparsity",
        [
            Sparsity(mode=SparsityMode.MAX, grid_field=DISC_FULL),
            Sparsity(grid_field=DISC_FULL),
            Sparsity(mode=SparsityMode.MAX, grid_field=ELLIPSE_FULL),
        ],
    ),
]

In [ ]:
fig = plot_biomarker_grid(biomarker_rows_part1, ncols=ncols, dpi=fig_dpi)
if figs_path is not None:
    for ext in ("png", "pdf"):
        output_path = figs_path / f"biomarkers_combined_part1.{ext}"
        fig.savefig(output_path, format=ext, dpi=fig_dpi, bbox_inches="tight")
        print(f"Saved {output_path}")

In [ ]:
fig = plot_biomarker_grid(biomarker_rows_part2, ncols=ncols, dpi=fig_dpi)
if figs_path is not None:
    for ext in ("png", "pdf"):
        output_path = figs_path / f"biomarkers_combined_part2.{ext}"
        fig.savefig(output_path, format=ext, dpi=fig_dpi, bbox_inches="tight")
        print(f"Saved {output_path}")